# V7.1 — Test YOLO26m trên val và test cố định

Notebook tải V7.1 best từ Camber, đánh giá cả val/test và so trực tiếp với seed85 best trên cùng dataset. Kết quả per-class được lưu CSV/JSON và upload lại Camber.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, json, time
import pandas as pd

EXPERIMENT = 'finding_v7p1_171126'
RUN_NAME = 'yolo26m_finetune'
REMOTE_RUN = f'stash://bunpmc/projects/{EXPERIMENT}/{RUN_NAME}'
BASELINE_REMOTE = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'
IMGSZ, BATCH = 640, 8

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir(): return p
    raise RuntimeError('Mở notebook từ repo train')

ROOT = find_repo()
DATA_YAML = ROOT / 'benchmakring_transmission_line_defect_model/data/finding_nemo_human_reviewed_6cls_v2_171126/data.yaml'
OUT = ROOT / 'benchmakring_transmission_line_defect_model/results/finding_v7p1_171126'
WEIGHTS = ROOT / 'weights/v7p1'
OUT.mkdir(parents=True, exist_ok=True); WEIGHTS.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.is_file(), DATA_YAML
print('Dataset:', DATA_YAML)
print('Output :', OUT)

In [ ]:
# %pip install -U "ultralytics==8.4.154" pandas python-dotenv
import torch, ultralytics
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError: pass
assert torch.cuda.is_available(), 'Không thấy CUDA'
print('Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0))

def find_camber():
    for x in (os.environ.get('CAMBER_BIN'), shutil.which('camber'), str(Path.home()/'.camber/bin/camber.exe'), str(Path.home()/'.camber/bin/camber')):
        if x and Path(x).is_file(): return str(Path(x))
    raise RuntimeError('Không tìm thấy Camber CLI')
CAMBER = find_camber()
def camber(*parts, check=True):
    p = subprocess.run([CAMBER, 'stash', *map(str, parts)], env=os.environ.copy(), capture_output=True, text=True)
    if check and p.returncode: raise RuntimeError((p.stderr or p.stdout)[-2000:])
    return p
assert subprocess.run([CAMBER, 'me'], env=os.environ.copy(), capture_output=True).returncode == 0
print('Camber authentication OK')

## Tải hai checkpoint

In [ ]:
MODELS = {
    'seed85_before': (BASELINE_REMOTE, WEIGHTS / 'seed85_curated_yolo26m_best.pt'),
    'v7p1_after': (REMOTE_RUN + '/weights/best.pt', WEIGHTS / 'v7p1_yolo26m_best.pt'),
}
for name, (remote, local) in MODELS.items():
    if not local.is_file():
        print('Downloading', name, '...')
        camber('cp', remote, local)
    print(name, round(local.stat().st_size / 2**20, 1), 'MiB')

## Chạy val + test và lưu per-class CSV

In [ ]:
summary_rows, class_rows = [], []
for model_name, (_, checkpoint) in MODELS.items():
    model = YOLO(str(checkpoint))
    for split in ('val', 'test'):
        print(f'\n===== {model_name} / {split} =====')
        metrics = model.val(
            data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH, device=0,
            project=str(OUT / 'runs'), name=f'{model_name}_{split}', exist_ok=True,
            plots=True, save_json=False, verbose=True,
        )
        summary_rows.append({
            'model': model_name, 'split': split, 'precision': float(metrics.box.mp),
            'recall': float(metrics.box.mr), 'mAP50': float(metrics.box.map50),
            'mAP50_95': float(metrics.box.map),
        })
        names = metrics.names
        for i, cls in enumerate(metrics.box.ap_class_index.astype(int).tolist()):
            class_rows.append({
                'model': model_name, 'split': split, 'class_id': cls, 'class': names[cls],
                'precision': float(metrics.box.p[i]), 'recall': float(metrics.box.r[i]),
                'AP50': float(metrics.box.ap50[i]), 'AP50_95': float(metrics.box.ap[i]),
            })

summary = pd.DataFrame(summary_rows)
per_class = pd.DataFrame(class_rows)
summary.to_csv(OUT / 'benchmark_summary.csv', index=False)
per_class.to_csv(OUT / 'per_class_summary.csv', index=False)
(OUT / 'benchmark_results.json').write_text(json.dumps({
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': str(DATA_YAML), 'summary': summary_rows, 'per_class': class_rows,
}, indent=2), encoding='utf-8')
display(summary)
display(per_class)

## Chênh lệch V7.1 so với seed85 và upload kết quả

In [ ]:
before = summary[summary.model == 'seed85_before'].set_index('split')
after = summary[summary.model == 'v7p1_after'].set_index('split')
delta = (after[['precision','recall','mAP50','mAP50_95']] - before[['precision','recall','mAP50','mAP50_95']]).reset_index()
delta.insert(0, 'comparison', 'v7p1_after-minus-seed85_before')
delta.to_csv(OUT / 'delta_vs_seed85.csv', index=False)
display(delta)

REMOTE_EVAL = REMOTE_RUN + '/evaluation'
if camber('ls', REMOTE_EVAL, check=False).returncode != 0:
    camber('mkdir', REMOTE_EVAL, check=False)
for filename in ('benchmark_summary.csv', 'per_class_summary.csv', 'delta_vs_seed85.csv', 'benchmark_results.json'):
    camber('cp', OUT / filename, REMOTE_EVAL + '/' + filename)
print('Uploaded:', REMOTE_EVAL)